In [ ]:
%cd ../src

In [ ]:
import polars as pl

from config.brokers import KiteConfig
from config.compute import NSEBasePopConfig

In [ ]:
NSEBasePopConfig.ADDV

In [ ]:
NSEBasePopConfig.IPO_ADDV

In [ ]:
NSEBasePopConfig.STOCKS_ADDV

In [ ]:
_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)

In [ ]:
data = pl.read_database_uri(
    query=f"SELECT * from {_tbl_name}", uri=KiteConfig.DB_CONN, engine="connectorx"
)

In [ ]:
_max_timestamp = (
    data.lazy()
    .with_columns(pl.col("timestamp").cast(pl.Date()))
    .select("timestamp")
    .max()
    .collect()
    .item(0, 0)
)

df = (
    data.lazy()
    .with_columns(pl.col("timestamp").cast(pl.Date()))
    .with_columns(
        # Dollar Volume
        [
            (
                (pl.col("close") * pl.col("volume")).rolling_mean(
                    window_size=NSEBasePopConfig.ADDV
                )
            )
            .round(2)
            .over(
                partition_by="symbol",
                order_by="timestamp",
                descending=False,
            )
            .alias(f"dollar_volume_sma_{NSEBasePopConfig.ADDV}")
        ]
    )
)

In [ ]:
ipo_stocks_list = (
    df.filter(
        pl.col(f"dollar_volume_sma_{NSEBasePopConfig.ADDV}").is_null()
        & (pl.col("timestamp") == _max_timestamp)
    )
    .sort("symbol")
    .collect()
    .get_column("symbol")
    .to_list()
)

In [ ]:
ipo_stocks_df = (
    df.filter(pl.col("symbol").is_in(ipo_stocks_list))
    .group_by("symbol")
    .agg(
        pl.col("timestamp").max().alias("timestamp"),
        (pl.col("close") * pl.col("volume")).mean().round(2).alias("avg_dollar_volume"),
        pl.col("timestamp").len().alias("days_since_ipo"),
    )
    .filter(pl.col("avg_dollar_volume") >= NSEBasePopConfig.IPO_ADDV)
    .with_columns(pl.lit(True).alias("ipo_pop_flag"))
    .sort("symbol")
    .collect()
)

In [ ]:
base_pop_df = (
    df.filter(
        (pl.col("timestamp") == _max_timestamp)
        & (
            pl.col(f"dollar_volume_sma_{NSEBasePopConfig.ADDV}")
            >= NSEBasePopConfig.STOCKS_ADDV
        )
    )
    .rename({f"dollar_volume_sma_{NSEBasePopConfig.ADDV}": "avg_dollar_volume"})
    .select("symbol", "timestamp", "avg_dollar_volume")
    .with_columns(
        pl.lit(None).alias("days_since_ipo"), pl.lit(False).alias("ipo_pop_flag")
    )
    .collect()
)

In [ ]:
ipo_stocks_df.columns == base_pop_df.columns

In [ ]:
ipo_stocks_df.vstack(base_pop_df).write_parquet("base_population.parquet")